In [1]:
import json
import duckdb
import pandas as pd
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_DIR = BASE_DIR / "data_sets"
RESULT_DIR = BASE_DIR / "results"
ANALYSIS_DIR = BASE_DIR / "analysis"

DB_PATH = ANALYSIS_DIR / "step02_05_alignment_diagnosis.duckdb"
con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

delay_cleaned_path = RESULT_DIR / "combined_delay_cleaned.csv"
sched_imputed_path = RESULT_DIR / "combined_schedule_imputed.csv"
raw_delay_path = DATA_DIR / "combined_delay.csv"

con.execute(f"""
CREATE OR REPLACE VIEW delay_cleaned AS
SELECT 
    CAST(date AS DATE) AS date,
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(delay AS DOUBLE) AS delay,
    CAST(is_imputed AS BOOLEAN) AS is_imputed
FROM read_csv_auto('{delay_cleaned_path.as_posix()}', header=True);

CREATE OR REPLACE VIEW schedule_imputed AS
SELECT 
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(distance_from_origin AS DOUBLE) AS distance_from_origin,
    CAST(arrival_day AS INTEGER) AS arrival_day,
    CAST(arrival_time AS VARCHAR) AS arrival_time,
    CAST(departure_day AS INTEGER) AS departure_day,
    CAST(departure_time AS VARCHAR) AS departure_time
FROM read_csv_auto('{sched_imputed_path.as_posix()}', header=True);

CREATE OR REPLACE VIEW raw_delay AS
SELECT 
    CAST(date AS DATE) AS date,
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name
FROM read_csv_auto('{raw_delay_path.as_posix()}', header=True);
""")

# Collapse delay_cleaned into distinct stops weighted by row count (total, observed, imputed)
con.execute("""
CREATE OR REPLACE TABLE delay_stops AS
SELECT 
    train_no,
    station_no AS delay_station_no,
    station_name AS delay_station_name,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS n_observed_rows,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS n_imputed_rows
FROM delay_cleaned
GROUP BY train_no, station_no, station_name;
""")

stability_df = q("""
SELECT 
    COUNT(*) AS distinct_delay_stops,
    SUM(n_rows) AS total_rows,
    (SELECT COUNT(*) FROM (
        SELECT train_no, delay_station_no
        FROM delay_stops
        GROUP BY train_no, delay_station_no
        HAVING COUNT(DISTINCT delay_station_name) > 1
    )) AS unstable_stops_across_dates
FROM delay_stops;
""")

print("=== STOP COLLAPSE & TEMPORAL STABILITY CHECK ===")
print(stability_df.to_string(index=False))
FINDINGS["stop_stability"] = stability_df.iloc[0].to_dict()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== STOP COLLAPSE & TEMPORAL STABILITY CHECK ===
 distinct_delay_stops  total_rows  unstable_stops_across_dates
               141646  36614824.0                            0


In [2]:
# ==========================================
# STEP 1: CLASSIFICATION OF EVERY DELAY STOP
# ==========================================
print("\n=== STEP 1: CLASSIFICATION OF DELAY STOPS ===")

con.execute("""
CREATE OR REPLACE TABLE classified_stops AS
WITH TrainInSched AS (
    SELECT DISTINCT train_no FROM schedule_imputed
),
SameStopMatch AS (
    SELECT 
        d.train_no,
        d.delay_station_no,
        d.delay_station_name,
        d.n_rows,
        d.n_observed_rows,
        d.n_imputed_rows,
        ts.train_no IS NOT NULL AS train_in_sched,
        s_same.station_no AS same_sched_station_no,
        s_same.station_name AS same_sched_station_name
    FROM delay_stops d
    LEFT JOIN TrainInSched ts ON d.train_no = ts.train_no
    LEFT JOIN schedule_imputed s_same 
      ON d.train_no = s_same.train_no 
     AND d.delay_station_no = s_same.station_no
),
NameMatchInTrain AS (
    SELECT 
        m.*,
        COUNT(s_name.station_no) AS n_name_matches_in_train,
        MIN(s_name.station_no) AS matched_sched_station_no,
        MIN(CASE WHEN UPPER(TRIM(m.delay_station_name)) = UPPER(TRIM(s_name.station_name)) 
                  AND m.delay_station_name != s_name.station_name THEN 1 ELSE 0 END) AS is_format_diff
    FROM SameStopMatch m
    LEFT JOIN schedule_imputed s_name
      ON m.train_no = s_name.train_no
     AND UPPER(TRIM(m.delay_station_name)) = UPPER(TRIM(s_name.station_name))
    GROUP BY 
        m.train_no, m.delay_station_no, m.delay_station_name, 
        m.n_rows, m.n_observed_rows, m.n_imputed_rows,
        m.train_in_sched, m.same_sched_station_no, m.same_sched_station_name
)
SELECT 
    train_no,
    delay_station_no,
    delay_station_name,
    n_rows,
    n_observed_rows,
    n_imputed_rows,
    same_sched_station_no,
    same_sched_station_name,
    n_name_matches_in_train,
    matched_sched_station_no,
    CASE
        WHEN NOT train_in_sched THEN 'train_not_in_schedule'
        WHEN same_sched_station_no IS NOT NULL AND delay_station_name = same_sched_station_name THEN 'exact'
        WHEN same_sched_station_no IS NOT NULL AND UPPER(TRIM(delay_station_name)) = UPPER(TRIM(same_sched_station_name)) THEN 'format_only'
        WHEN n_name_matches_in_train >= 1 AND matched_sched_station_no != delay_station_no THEN 'shifted'
        WHEN same_sched_station_no IS NOT NULL AND n_name_matches_in_train = 0 THEN 'different_station'
        ELSE 'orphan_name_not_in_schedule'
    END AS cls,
    CASE 
        WHEN n_name_matches_in_train = 1 THEN (matched_sched_station_no - delay_station_no)
        ELSE NULL
    END AS shift_offset
FROM NameMatchInTrain;
""")

cls_summary_df = q("""
SELECT 
    cls,
    COUNT(*) AS n_stops,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS pct_stops,
    SUM(n_rows) AS n_rows,
    ROUND(SUM(n_rows) * 100.0 / SUM(SUM(n_rows)) OVER (), 4) AS pct_rows,
    SUM(n_observed_rows) AS n_observed_rows,
    SUM(n_imputed_rows) AS n_imputed_rows
FROM classified_stops
GROUP BY cls
ORDER BY n_rows DESC;
""")
print(cls_summary_df.to_string(index=False))
FINDINGS["step1_classification"] = cls_summary_df.to_dict(orient="records")


=== STEP 1: CLASSIFICATION OF DELAY STOPS ===
              cls  n_stops  pct_stops     n_rows  pct_rows  n_observed_rows  n_imputed_rows
            exact   112948    79.7396 30018784.0   81.9853       29661304.0        357480.0
          shifted    24495    17.2931  5737461.0   15.6698        5618694.0        118767.0
different_station     4203     2.9673   858579.0    2.3449         302320.0        556259.0


In [3]:
# ==========================================
# STEP 2: ROOT CAUSE (SHIFT OFFSETS & ROUTE LENGTHS)
# ==========================================
print("\n=== STEP 2A: SHIFT OFFSET DISTRIBUTION (sched_station_no - delay_station_no) ===")

shift_dist_df = q("""
SELECT 
    shift_offset,
    COUNT(*) AS n_stops,
    SUM(n_rows) AS n_rows,
    ROUND(SUM(n_rows) * 100.0 / (SELECT SUM(n_rows) FROM classified_stops WHERE cls = 'shifted'), 4) AS pct_of_shifted_rows
FROM classified_stops
WHERE cls = 'shifted'
GROUP BY shift_offset
ORDER BY n_rows DESC
LIMIT 15;
""")
print(shift_dist_df.to_string(index=False))

print("\n=== STEP 2B: TRAIN-LEVEL AFFECTED SHARE BREAKDOWN ===")
train_impact_df = q("""
WITH TrainStopStats AS (
    SELECT 
        train_no,
        COUNT(*) AS total_stops,
        COUNT(*) FILTER (WHERE cls = 'exact') AS exact_stops,
        COUNT(*) FILTER (WHERE cls != 'exact') AS affected_stops,
        CAST(COUNT(*) FILTER (WHERE cls != 'exact') AS DOUBLE) / COUNT(*) AS affected_ratio
    FROM classified_stops
    GROUP BY train_no
)
SELECT 
    COUNT(*) AS total_trains_in_delay,
    COUNT(*) FILTER (WHERE affected_stops = 0) AS fully_clean_trains,
    COUNT(*) FILTER (WHERE affected_stops > 0 AND affected_ratio < 0.5) AS under_half_affected_trains,
    COUNT(*) FILTER (WHERE affected_ratio >= 0.5 AND affected_stops < total_stops) AS over_half_affected_trains,
    COUNT(*) FILTER (WHERE affected_stops = total_stops) AS fully_affected_trains
FROM TrainStopStats;
""")
print(train_impact_df.to_string(index=False))

print("\n=== STEP 2C: ROUTE-LENGTH COMPARISON (RAW DELAY vs SCHEDULE & CLEANED DELAY vs SCHEDULE) ===")
route_len_df = q("""
WITH RawTrainLen AS (
    SELECT train_no, MAX(station_no) AS raw_delay_max_stop, COUNT(DISTINCT station_no) AS raw_delay_n_stops
    FROM raw_delay
    GROUP BY train_no
),
CleanedTrainLen AS (
    SELECT train_no, MAX(delay_station_no) AS cleaned_delay_max_stop, COUNT(DISTINCT delay_station_no) AS cleaned_delay_n_stops
    FROM delay_stops
    GROUP BY train_no
),
SchedTrainLen AS (
    SELECT train_no, MAX(station_no) AS sched_max_stop, COUNT(DISTINCT station_no) AS sched_n_stops
    FROM schedule_imputed
    GROUP BY train_no
)
SELECT 
    COUNT(*) AS total_raw_delay_trains,
    COUNT(*) FILTER (WHERE s.train_no IS NOT NULL AND r.raw_delay_max_stop = s.sched_max_stop) AS raw_identical_route_len_trains,
    COUNT(*) FILTER (WHERE s.train_no IS NOT NULL AND (r.raw_delay_max_stop - s.sched_max_stop) BETWEEN 1 AND 8) AS raw_delay_1_to_8_stops_longer,
    COUNT(*) FILTER (WHERE s.train_no IS NOT NULL AND (r.raw_delay_max_stop - s.sched_max_stop) > 8) AS raw_delay_gt_8_stops_longer,
    COUNT(*) FILTER (WHERE s.train_no IS NOT NULL AND r.raw_delay_max_stop < s.sched_max_stop) AS raw_delay_shorter_than_sched,
    COUNT(*) FILTER (WHERE s.train_no IS NULL) AS raw_train_not_in_sched,
    COUNT(*) FILTER (WHERE c.train_no IS NOT NULL AND s.train_no IS NOT NULL AND c.cleaned_delay_max_stop = s.sched_max_stop) AS cleaned_identical_route_len_trains
FROM RawTrainLen r
LEFT JOIN SchedTrainLen s ON r.train_no = s.train_no
LEFT JOIN CleanedTrainLen c ON r.train_no = c.train_no;
""")
print(route_len_df.to_string(index=False))

FINDINGS["step2_root_cause"] = {
    "shift_offset_distribution": shift_dist_df.to_dict(orient="records"),
    "train_impact_breakdown": train_impact_df.iloc[0].to_dict(),
    "route_length_comparison": route_len_df.iloc[0].to_dict()
}


=== STEP 2A: SHIFT OFFSET DISTRIBUTION (sched_station_no - delay_station_no) ===
 shift_offset  n_stops    n_rows  pct_of_shifted_rows
           -1    15944 3596911.0              62.6917
           -2     4402 1073818.0              18.7159
           -3     1413  366257.0               6.3836
           -4      622  168093.0               2.9297
            1      626  145450.0               2.5351
           -5      400   99158.0               1.7283
           -6      263   63481.0               1.1064
           -7      171   57446.0               1.0012
           -8      187   56162.0               0.9789
            2       81   24627.0               0.4292
          -10       48   14764.0               0.2573
            5       37   12260.0               0.2137
          -15       30   10602.0               0.1848
           -9       35    6484.0               0.1130
           20       16    5423.0               0.0945

=== STEP 2B: TRAIN-LEVEL AFFECTED SHARE BREAKDOWN ===

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_raw_delay_trains  raw_identical_route_len_trains  raw_delay_1_to_8_stops_longer  raw_delay_gt_8_stops_longer  raw_delay_shorter_than_sched  raw_train_not_in_sched  cleaned_identical_route_len_trains
                   7033                            4902                           1918                           95                            85                      33                                6914


In [4]:
# ==========================================
# STEP 3: BUILD & VALIDATE stop_alignment_map
# ==========================================
print("\n=== STEP 3: BUILDING & VALIDATING stop_alignment_map.csv ===")

con.execute("""
CREATE OR REPLACE TABLE candidate_alignment AS
SELECT 
    train_no,
    delay_station_no,
    delay_station_name,
    n_rows,
    n_observed_rows,
    n_imputed_rows,
    cls,
    n_name_matches_in_train,
    CASE 
        WHEN cls IN ('exact', 'format_only') THEN same_sched_station_no
        WHEN cls = 'shifted' AND n_name_matches_in_train = 1 THEN matched_sched_station_no
        ELSE NULL
    END AS candidate_sched_station_no
FROM classified_stops;

CREATE OR REPLACE TABLE stop_alignment_map AS
WITH CollisionCheck AS (
    SELECT 
        train_no,
        candidate_sched_station_no,
        COUNT(*) AS n_delay_stops_mapping_here
    FROM candidate_alignment
    WHERE candidate_sched_station_no IS NOT NULL
    GROUP BY train_no, candidate_sched_station_no
)
SELECT 
    c.train_no,
    c.delay_station_no,
    CASE 
        WHEN cc.n_delay_stops_mapping_here = 1 THEN c.candidate_sched_station_no
        ELSE NULL
    END AS sched_station_no,
    CASE 
        WHEN cc.n_delay_stops_mapping_here > 1 THEN 'collision_dropped'
        WHEN c.cls = 'shifted' AND c.n_name_matches_in_train > 1 THEN 'ambiguous_name_dropped'
        ELSE c.cls
    END AS cls
FROM candidate_alignment c
LEFT JOIN CollisionCheck cc
  ON c.train_no = cc.train_no
 AND c.candidate_sched_station_no = cc.candidate_sched_station_no
ORDER BY c.train_no, c.delay_station_no;
""")

alignment_audit_df = q("""
SELECT 
    COUNT(*) AS total_stops_in_map,
    COUNT(*) FILTER (WHERE m.sched_station_no IS NOT NULL) AS aligned_stops,
    COUNT(*) FILTER (WHERE m.sched_station_no IS NULL) AS unaligned_stops,
    SUM(d.n_rows) AS total_rows,
    SUM(d.n_rows) FILTER (WHERE m.sched_station_no IS NOT NULL) AS aligned_rows,
    ROUND(SUM(d.n_rows) FILTER (WHERE m.sched_station_no IS NOT NULL) * 100.0 / SUM(d.n_rows), 4) AS aligned_rows_pct,
    SUM(d.n_rows) FILTER (WHERE m.sched_station_no IS NULL) AS unaligned_rows,
    ROUND(SUM(d.n_rows) FILTER (WHERE m.sched_station_no IS NULL) * 100.0 / SUM(d.n_rows), 4) AS unaligned_rows_pct,
    COUNT(*) FILTER (WHERE c.n_name_matches_in_train > 1) AS ambiguous_name_matches,
    COUNT(*) FILTER (WHERE m.cls = 'collision_dropped') AS collision_stops
FROM stop_alignment_map m
JOIN delay_stops d 
  ON m.train_no = d.train_no AND m.delay_station_no = d.delay_station_no
JOIN candidate_alignment c
  ON m.train_no = c.train_no AND m.delay_station_no = c.delay_station_no;
""")
print(alignment_audit_df.to_string(index=False))

# Verify 0 duplicate (train_no, sched_station_no) among aligned stops
dup_aligned_check = q("""
SELECT COUNT(*) AS duplicate_target_sched_stops
FROM (
    SELECT train_no, sched_station_no
    FROM stop_alignment_map
    WHERE sched_station_no IS NOT NULL
    GROUP BY train_no, sched_station_no
    HAVING COUNT(*) > 1
);
""")
print("\nDuplicate (train_no, sched_station_no) targets in map:", dup_aligned_check["duplicate_target_sched_stops"][0])

# Export stop_alignment_map.csv
map_file = RESULT_DIR / "stop_alignment_map.csv"
con.execute(f"COPY stop_alignment_map TO '{map_file.as_posix()}' (HEADER, DELIMITER ',');")
print(f"-> Saved stop_alignment_map to {map_file.as_posix()}")

FINDINGS["step3_alignment_resolution"] = {
    **alignment_audit_df.iloc[0].to_dict(),
    "duplicate_target_sched_stops": int(dup_aligned_check["duplicate_target_sched_stops"][0])
}

# Save findings JSON
findings_file = RESULT_DIR / "step02_05_alignment_FINDINGS.json"
with open(findings_file, "w") as f:
    json.dump(FINDINGS, f, indent=4)
print(f"-> Saved findings to {findings_file.as_posix()}")

con.close()


=== STEP 3: BUILDING & VALIDATING stop_alignment_map.csv ===
 total_stops_in_map  aligned_stops  unaligned_stops  total_rows  aligned_rows  aligned_rows_pct  unaligned_rows  unaligned_rows_pct  ambiguous_name_matches  collision_stops
             141646         137443             4203  36614824.0    35756245.0           97.6551        858579.0              2.3449                       0                0

Duplicate (train_no, sched_station_no) targets in map: 0
-> Saved stop_alignment_map to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/stop_alignment_map.csv
-> Saved findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step02_05_alignment_FINDINGS.json
